# Feature selection – Titanic – Bouckaert John & Hugo Fiévet

## Introduction

Dans ce notebook, nous réalisons la sélection des variables à partir des jeux de données prétraités obtenus dans le notebook précédent.

Conformément aux consignes, nous comparons trois approches de sélection :
- une sélection personnelle fondée sur l’analyse exploratoire réalisée en phase 1 ;
- une sélection automatique par Variance Threshold ;
- une sélection automatique par SelectKBest.

L’objectif est de construire plusieurs sous-ensembles de variables exploitables pour les étapes de modélisation, puis de sauvegarder les fichiers CSV correspondants.

## Table of contents

1. [Loading the preprocessed datasets](#1-loading-the-preprocessed-datasets)  
2. [Personal feature selection](#2-personal-feature-selection)  
3. [Feature selection with Variance Threshold](#3-feature-selection-with-variance-threshold)  
4. [Feature selection with SelectKBest](#4-feature-selection-with-selectkbest)  
5. [Saving the selected datasets](#5-saving-the-selected-datasets)  
6. [Critical discussion](#6-critical-discussion)  
7. [Conclusion](#conclusion)


## 1. Loading the preprocessed datasets

Dans cette première étape, nous chargeons les fichiers produits à l’issue du preprocessing.  
Ces fichiers constituent la base de travail du présent notebook et seront utilisés pour appliquer les différentes méthodes de sélection de variables.

L’objectif est de vérifier que les dimensions des jeux d’entraînement et de test sont cohérentes avant de commencer les sélections.

In [37]:
import pandas as pd

# Chargement des jeux de données prétraités
X_train = pd.read_csv("xtrain.csv")
X_test = pd.read_csv("xtest.csv")
y_train = pd.read_csv("ytrain.csv")
y_test = pd.read_csv("ytest.csv")

print("Dimensions de X_train :", X_train.shape)
print("Dimensions de X_test :", X_test.shape)
print("Dimensions de y_train :", y_train.shape)
print("Dimensions de y_test :", y_test.shape)

print("\nColonnes de X_train :")
print(X_train.columns.tolist())

print("\nAperçu de X_train :")
print(X_train.head())

Dimensions de X_train : (1047, 22)
Dimensions de X_test : (262, 22)
Dimensions de y_train : (1047, 1)
Dimensions de y_test : (262, 1)

Colonnes de X_train :
['age', 'sibsp', 'parch', 'fare', 'has_cabin', 'family_size', 'pclass_1', 'pclass_2', 'pclass_3', 'sex_female', 'sex_male', 'embarked_C', 'embarked_Q', 'embarked_S', 'title_grouped_Master', 'title_grouped_Miss', 'title_grouped_Mr', 'title_grouped_Mrs', 'title_grouped_Other', 'family_group_alone', 'family_group_large_family', 'family_group_small_family']

Aperçu de X_train :
        age     sibsp     parch      fare  has_cabin  family_size  pclass_1  \
0 -0.820265 -0.479571 -0.447597 -0.499246          0    -0.557940         0   
1 -0.371539  0.510785 -0.447597 -0.090614          0     0.083293         0   
2 -1.343779  3.481854  1.872374 -0.017890          0     3.289456         0   
3 -0.296752 -0.479571 -0.447597 -0.509988          0    -0.557940         0   
4 -0.969840 -0.479571 -0.447597 -0.501292          0    -0.557940      

Les fichiers issus du preprocessing ont bien été chargés et leurs dimensions sont cohérentes. Les jeux de données explicatifs contiennent respectivement **1047 observations et 22 variables** pour `X_train`, et **262 observations et 22 variables** pour `X_test`. Les variables cibles `y_train` et `y_test` ont également été récupérées correctement.

La liste des colonnes confirme que le preprocessing a bien produit un dataset entièrement numérique. Les variables quantitatives standardisées (`age`, `sibsp`, `parch`, `fare`, `family_size`) coexistent désormais avec des variables binaires issues de l’encodage catégoriel, comme `pclass_1`, `sex_female`, `title_grouped_Mr` ou encore les nouvelles colonnes liées à `family_group`.

L’ajout de `family_group_alone`, `family_group_large_family` et `family_group_small_family` traduit directement une conclusion importante de la phase 1 : la relation entre la structure familiale et la survie n’apparaissait pas strictement linéaire. Cette nouvelle représentation permettra de vérifier si cette information regroupée est mieux captée par les méthodes de sélection de variables que la seule variable numérique `family_size`.

L’aperçu de `X_train` montre donc que les jeux de données sont prêts à être utilisés pour la sélection de variables. Le notebook peut maintenant se concentrer sur la comparaison des trois approches prévues : la sélection personnelle, la méthode Variance Threshold et la méthode SelectKBest.

## 2. Personal feature selection

Dans cette étape, nous définissons une sélection personnelle de variables à partir des conclusions obtenues lors de la phase 1.

L’objectif est de conserver les variables qui semblaient les plus pertinentes pour expliquer la survie, tout en écartant les variables jugées peu informatives ou redondantes.

In [38]:
# Sélection personnelle fondée sur la phase 1
selected_features_P = [
    "fare",
    "has_cabin",
    "pclass_1", "pclass_2", "pclass_3",
    "sex_female", "sex_male",
    "title_grouped_Master", "title_grouped_Miss",
    "title_grouped_Mr", "title_grouped_Mrs", "title_grouped_Other",
    "family_group_alone", "family_group_large_family", "family_group_small_family"
]

X_train_P = X_train[selected_features_P].copy()
X_test_P = X_test[selected_features_P].copy()

print("Variables retenues pour la sélection personnelle :")
print(selected_features_P)

print("\nDimensions de X_train_P :")
print(X_train_P.shape)

print("\nDimensions de X_test_P :")
print(X_test_P.shape)

print("\nAperçu de X_train_P :")
print(X_train_P.head())

Variables retenues pour la sélection personnelle :
['fare', 'has_cabin', 'pclass_1', 'pclass_2', 'pclass_3', 'sex_female', 'sex_male', 'title_grouped_Master', 'title_grouped_Miss', 'title_grouped_Mr', 'title_grouped_Mrs', 'title_grouped_Other', 'family_group_alone', 'family_group_large_family', 'family_group_small_family']

Dimensions de X_train_P :
(1047, 15)

Dimensions de X_test_P :
(262, 15)

Aperçu de X_train_P :
       fare  has_cabin  pclass_1  pclass_2  pclass_3  sex_female  sex_male  \
0 -0.499246          0         0         0         1           1         0   
1 -0.090614          0         0         1         0           1         0   
2 -0.017890          0         0         0         1           1         0   
3 -0.509988          0         0         0         1           0         1   
4 -0.501292          0         0         0         1           1         0   

   title_grouped_Master  title_grouped_Miss  title_grouped_Mr  \
0                     0                   1 

La sélection personnelle retient **15 variables** jugées les plus pertinentes à partir des conclusions de la phase 1. Les jeux `X_train_P` et `X_test_P` conservent une structure cohérente, avec respectivement **1047 observations** et **262 observations**, et le même nombre de variables dans les deux cas.

Les variables retenues correspondent aux facteurs qui semblaient les plus liés à la survie lors de l’analyse exploratoire. On retrouve d’abord des éléments fortement discriminants comme le **sexe** et la **classe**, ici représentés par les variables binaires `sex_female`, `sex_male`, `pclass_1`, `pclass_2` et `pclass_3`. Le **titre** extrait du nom a également été conservé à travers les variables `title_grouped_*`, car il apportait une information sociale utile sur le profil des passagers.

La variable `has_cabin` a aussi été retenue, car le fait qu’une cabine soit renseignée ou non semblait porter une information indirecte sur le statut du passager. La variable `fare` a également été conservée, puisqu’elle traduisait en partie le niveau social et la classe du passager.

Enfin, la structure familiale est représentée ici par les variables issues de `family_group` plutôt que par la seule variable numérique `family_size`. Ce choix est cohérent avec la phase 1, qui suggérait une relation non linéaire entre la taille de la famille et la survie. La distinction entre passagers seuls, petites familles et grandes familles permet donc de mieux refléter les profils observés lors de l’analyse exploratoire.

À l’inverse, certaines variables ont été volontairement écartées de cette sélection personnelle, comme `age`, `sibsp`, `parch`, `embarked` et `family_size`, car elles semblaient soit peu informatives, soit moins adaptées que des variables dérivées ou regroupées.

## 3. Feature selection with Variance Threshold

Dans cette étape, nous appliquons une sélection automatique de variables basée sur la variance.  
Le principe de cette méthode est d’éliminer les variables dont la variance est trop faible, car elles apportent peu d’information et risquent de nuire à la qualité du modèle.

Avant de fixer un seuil, nous commençons par examiner la variance de chaque variable dans le jeu d’entraînement.

In [39]:
# Calcul de la variance de chaque variable
variances = X_train.var().sort_values()

print("Variance de chaque variable :")
print(variances.round(2))

print("\nRésumé des variances :")
print(variances.describe().round(2))

Variance de chaque variable :
title_grouped_Other          0.02
title_grouped_Master         0.05
family_group_large_family    0.06
embarked_Q                   0.09
title_grouped_Mrs            0.13
title_grouped_Miss           0.16
embarked_C                   0.16
pclass_2                     0.17
has_cabin                    0.17
pclass_1                     0.18
embarked_S                   0.21
family_group_small_family    0.22
sex_male                     0.23
sex_female                   0.23
family_group_alone           0.24
title_grouped_Mr             0.24
pclass_3                     0.25
parch                        1.00
family_size                  1.00
fare                         1.00
age                          1.00
sibsp                        1.00
dtype: float64

Résumé des variances :
count    22.00
mean      0.35
std       0.36
min       0.02
25%       0.16
50%       0.21
75%       0.25
max       1.00
dtype: float64


L’examen des variances montre que les variables du jeu d’entraînement ne contribuent pas toutes de la même manière à la dispersion des données. Les variables présentant les variances les plus faibles sont `title_grouped_Other`, `title_grouped_Master`, `family_group_large_family` et `embarked_Q`. Cela signifie qu’elles prennent relativement rarement la valeur `1` dans le dataset et qu’elles varient donc peu d’un passager à l’autre.

L’ajout de `family_group` apporte ici une information utile. On observe que `family_group_large_family` a une variance faible, ce qui est cohérent avec le fait que les grandes familles restent peu fréquentes dans les données. En revanche, `family_group_small_family` et `family_group_alone` présentent des variances plus élevées, ce qui montre que ces deux profils sont bien plus représentés et mieux répartis dans le dataset. Cette nouvelle variable regroupée semble donc plus exploitable statistiquement que certaines tailles exactes de famille prises isolément.

Comme précédemment, les variables quantitatives standardisées (`age`, `sibsp`, `parch`, `fare`, `family_size`) présentent toutes une variance proche de `1`, ce qui est normal après la standardisation. Les variables binaires les plus équilibrées, comme `sex_female`, `sex_male`, `family_group_alone`, `title_grouped_Mr` ou `pclass_3`, affichent également des variances plus élevées, signe qu’elles différencient davantage les passagers.

Ces résultats montrent qu’un seuil trop élevé risquerait encore d’éliminer des modalités rares mais potentiellement intéressantes. En particulier, même si `family_group_large_family` varie peu, cette variable peut rester pertinente sur le plan interprétatif. Il faudra donc choisir avec prudence le seuil du Variance Threshold afin de supprimer uniquement les variables les moins dispersées, sans appauvrir excessivement le jeu de données.

### Choice of the threshold

Au vu des variances observées, nous choisissons dans un premier temps un seuil de `0.05`.  
Ce choix permet de supprimer uniquement les variables les moins dispersées, c’est-à-dire celles qui prennent très rarement la valeur 1 ou qui varient très peu dans le dataset.

L’objectif est d’éliminer les variables les plus faibles selon ce critère, tout en conservant la majorité des variables potentiellement informatives.

In [40]:
from sklearn.feature_selection import VarianceThreshold

# Définition du seuil
vt_threshold = 0.05

# Application du Variance Threshold
selector_vt = VarianceThreshold(threshold=vt_threshold)
selector_vt.fit(X_train)

# Colonnes conservées
selected_features_VT = X_train.columns[selector_vt.get_support()].tolist()
removed_features_VT = X_train.columns[~selector_vt.get_support()].tolist()

# Création des nouveaux jeux de données
X_train_VT = X_train[selected_features_VT].copy()
X_test_VT = X_test[selected_features_VT].copy()

print("Seuil choisi pour Variance Threshold :", vt_threshold)

print("\nVariables conservées :")
print(selected_features_VT)

print("\nVariables supprimées :")
print(removed_features_VT)

print("\nDimensions de X_train_VT :")
print(X_train_VT.shape)

print("\nDimensions de X_test_VT :")
print(X_test_VT.shape)

Seuil choisi pour Variance Threshold : 0.05

Variables conservées :
['age', 'sibsp', 'parch', 'fare', 'has_cabin', 'family_size', 'pclass_1', 'pclass_2', 'pclass_3', 'sex_female', 'sex_male', 'embarked_C', 'embarked_Q', 'embarked_S', 'title_grouped_Miss', 'title_grouped_Mr', 'title_grouped_Mrs', 'family_group_alone', 'family_group_large_family', 'family_group_small_family']

Variables supprimées :
['title_grouped_Master', 'title_grouped_Other']

Dimensions de X_train_VT :
(1047, 20)

Dimensions de X_test_VT :
(262, 20)


Avec un seuil fixé à `0.05`, la méthode Variance Threshold conserve **20 variables** sur les **22** variables initiales. Les jeux `X_train_VT` et `X_test_VT` restent cohérents, avec respectivement **1047 observations** et **262 observations**, et le même nombre de colonnes dans les deux cas.

Seules deux variables ont été supprimées : `title_grouped_Master` et `title_grouped_Other`. Ce résultat est cohérent avec les variances observées précédemment. Ces deux variables correspondent à des modalités relativement rares dans le dataset et présentent donc une dispersion trop faible pour être conservées selon le seuil choisi.

À l’inverse, toutes les autres variables ont été retenues, y compris les variables quantitatives standardisées ainsi que les nouvelles variables liées à `family_group`. Cela montre que le seuil `0.05` reste prudent : il élimine uniquement les variables les moins dispersées, sans retirer des modalités rares mais encore interprétables comme `family_group_large_family`.

Cette sélection automatique diffère de la sélection personnelle, car elle repose uniquement sur la dispersion statistique des variables et non sur leur lien direct avec la survie. Elle constitue donc une approche complémentaire, plus mécanique, qui permet d’écarter les variables les plus faibles en termes de variance tout en conservant l’essentiel de l’information disponible.

## 4. Feature selection with SelectKBest

Dans cette étape, nous appliquons une sélection automatique de variables basée sur leur lien statistique avec la variable cible.

Contrairement à la méthode Variance Threshold, SelectKBest tient compte de la relation entre les variables explicatives et la variable `survived`. Comme certaines variables quantitatives ont été standardisées et peuvent prendre des valeurs négatives, nous utilisons ici le test ANOVA (`f_classif`), plus adapté que le test du chi carré dans ce contexte.

Avant de fixer le nombre de variables à conserver, nous examinons d’abord les scores obtenus par chaque variable.

In [41]:
from sklearn.feature_selection import SelectKBest, f_classif

# y sous forme unidimensionnelle
y_train_series = y_train.iloc[:, 0]

# Calcul des scores ANOVA
selector_kbest = SelectKBest(score_func=f_classif, k="all")
selector_kbest.fit(X_train, y_train_series)

scores_kbest = pd.DataFrame({
    "feature": X_train.columns,
    "score": selector_kbest.scores_,
    "p_value": selector_kbest.pvalues_
}).sort_values(by="score", ascending=False)

print("Scores obtenus par les variables avec SelectKBest (f_classif) :")
print(scores_kbest)

print("\nRésumé des scores :")
print(scores_kbest["score"].describe().round(2))

Scores obtenus par les variables avec SelectKBest (f_classif) :
                      feature       score       p_value
9                  sex_female  381.225175  1.266578e-72
10                   sex_male  381.225175  1.266578e-72
16           title_grouped_Mr  370.582840  6.407876e-71
17          title_grouped_Mrs  145.877418  1.559618e-31
21  family_group_small_family  101.409392  7.906507e-23
15         title_grouped_Miss   98.440959  3.105130e-22
4                   has_cabin   97.489454  4.818087e-22
6                    pclass_1   83.364258  3.439153e-19
8                    pclass_3   80.922710  1.080999e-18
3                        fare   68.159077  4.516066e-16
19         family_group_alone   57.711681  6.722728e-14
11                 embarked_C   46.270475  1.730859e-11
13                 embarked_S   26.698427  2.847471e-07
20  family_group_large_family   13.156453  3.003718e-04
2                       parch   11.448584  7.420382e-04
0                         age    2.72775

Les scores obtenus avec `SelectKBest` montrent quelles variables présentent le lien statistique le plus fort avec la variable cible `survived`. Contrairement à la méthode Variance Threshold, cette approche évalue directement la capacité de chaque variable à distinguer les survivants des non-survivants.

Les résultats confirment d’abord l’importance majeure du sexe et du profil social du passager. Les variables `sex_female` et `sex_male` obtiennent les scores les plus élevés, ce qui montre que le sexe est un facteur très fortement lié à la survie. Il est normal que ces deux variables aient exactement le même score et la même p-value, car elles sont complémentaires et traduisent la même information sous deux formes opposées. Les variables `title_grouped_Mr`, `title_grouped_Mrs` et `title_grouped_Miss` apparaissent également parmi les plus discriminantes, ce qui confirme que l’information extraite du nom reste très pertinente.

Les variables `has_cabin`, `pclass_1`, `pclass_3` et `fare` conservent elles aussi des scores élevés, ce qui reste cohérent avec les constats de la phase 1. Ces résultats confirment à nouveau l’importance du statut social, de la classe et des conditions de voyage dans l’explication de la survie.

La variable `family_group` joue ici un rôle intéressant. La modalité `family_group_small_family` obtient un score élevé et statistiquement très significatif, tandis que `family_group_alone` et `family_group_large_family` présentent également un lien significatif avec la survie. Ce résultat montre que la structure familiale regroupée est mieux captée par la méthode que la seule variable numérique `family_size`, dont le score reste faible et non significatif.

Enfin, on observe une rupture nette après `parch`. Les variables `age`, `title_grouped_Master`, `family_size`, `pclass_2`, `embarked_Q`, `sibsp` et `title_grouped_Other` obtiennent des scores faibles et des p-values supérieures à `0.05`, ce qui suggère une relation faible ou non significative avec la variable cible dans cette approche.

### Choice of k

Nous choisissons ici `k = 15`. Ce choix est fondé sur les résultats précédents : les 15 premières variables présentent toutes une p-value inférieure à `0.05`, ce qui indique une relation statistiquement significative avec la variable cible.

Ce choix est également cohérent avec la rupture observée dans les scores après `parch`. Nous conservons donc les variables les mieux classées par le test ANOVA, tout en excluant celles dont le lien avec la survie apparaît plus faible ou non significatif.

In [42]:
# Choix du nombre de variables à conserver
k_best_value = 15

selector_kbest_final = SelectKBest(score_func=f_classif, k=k_best_value)
selector_kbest_final.fit(X_train, y_train_series)

selected_features_KB = X_train.columns[selector_kbest_final.get_support()].tolist()
removed_features_KB = X_train.columns[~selector_kbest_final.get_support()].tolist()

X_train_KB = X_train[selected_features_KB].copy()
X_test_KB = X_test[selected_features_KB].copy()

print("Valeur de k choisie pour SelectKBest :", k_best_value)

print("\nVariables conservées :")
print(selected_features_KB)

print("\nVariables supprimées :")
print(removed_features_KB)

print("\nDimensions de X_train_KB :")
print(X_train_KB.shape)

print("\nDimensions de X_test_KB :")
print(X_test_KB.shape)

Valeur de k choisie pour SelectKBest : 15

Variables conservées :
['parch', 'fare', 'has_cabin', 'pclass_1', 'pclass_3', 'sex_female', 'sex_male', 'embarked_C', 'embarked_S', 'title_grouped_Miss', 'title_grouped_Mr', 'title_grouped_Mrs', 'family_group_alone', 'family_group_large_family', 'family_group_small_family']

Variables supprimées :
['age', 'sibsp', 'family_size', 'pclass_2', 'embarked_Q', 'title_grouped_Master', 'title_grouped_Other']

Dimensions de X_train_KB :
(1047, 15)

Dimensions de X_test_KB :
(262, 15)


Avec `k = 15`, la méthode `SelectKBest` conserve **15 variables** sur les **22** variables initiales. Les jeux `X_train_KB` et `X_test_KB` restent cohérents, avec respectivement **1047 observations** et **262 observations**, et le même nombre de colonnes dans les deux cas.

Les variables retenues correspondent aux mieux classées selon le test ANOVA appliqué à la relation entre les variables explicatives et la variable cible `survived`. On retrouve notamment `fare`, `has_cabin`, `pclass_1`, `pclass_3`, `sex_female`, `sex_male`, `embarked_C`, `embarked_S`, `title_grouped_Miss`, `title_grouped_Mr`, `title_grouped_Mrs`, ainsi que les trois variables issues de `family_group` et `parch`. Ces variables sont donc celles qui présentent ici le lien statistique le plus fort avec la survie.

À l’inverse, les variables `age`, `sibsp`, `family_size`, `pclass_2`, `embarked_Q`, `title_grouped_Master` et `title_grouped_Other` ont été supprimées. Ce résultat est cohérent avec les scores observés précédemment : ces variables présentaient des scores plus faibles et, pour plusieurs d’entre elles, une p-value supérieure à `0.05`, ce qui suggère une relation faible ou non significative avec la variable cible dans cette approche.

Cette sélection automatique apparaît globalement cohérente avec les constats de la phase 1. En particulier, le fait que `age`, `sibsp` et `family_size` ne soient pas retenues confirme qu’elles semblaient peu informatives sous leur forme brute. À l’inverse, la présence des trois variables issues de `family_group` montre que la structure familiale regroupée est mieux captée par la méthode que la seule variable numérique `family_size`.

## 5. Saving the selected datasets

Dans cette étape, nous sauvegardons les jeux de données obtenus après les trois méthodes de sélection de variables.

L’objectif est de produire des fichiers CSV distincts pour chaque approche afin de pouvoir les réutiliser facilement dans les notebooks de modélisation.

In [43]:
# Sauvegarde des jeux sélectionnés
X_train_P.to_csv("xtrain_P.csv", index=False)
X_test_P.to_csv("xtest_P.csv", index=False)

X_train_VT.to_csv("xtrain_VT.csv", index=False)
X_test_VT.to_csv("xtest_VT.csv", index=False)

X_train_KB.to_csv("xtrain_KB.csv", index=False)
X_test_KB.to_csv("xtest_KB.csv", index=False)

print("Dimensions des fichiers sauvegardés :")
print("X_train_P  :", X_train_P.shape)
print("X_test_P   :", X_test_P.shape)
print("X_train_VT :", X_train_VT.shape)
print("X_test_VT  :", X_test_VT.shape)
print("X_train_KB :", X_train_KB.shape)
print("X_test_KB  :", X_test_KB.shape)

Dimensions des fichiers sauvegardés :
X_train_P  : (1047, 15)
X_test_P   : (262, 15)
X_train_VT : (1047, 20)
X_test_VT  : (262, 20)
X_train_KB : (1047, 15)
X_test_KB  : (262, 15)


Les fichiers correspondant aux trois méthodes de sélection ont bien été sauvegardés. Chaque approche produit des jeux d’entraînement et de test cohérents, avec le même nombre de colonnes dans les deux sous-ensembles.

La sélection personnelle (`P`) conserve **15 variables**, la méthode `Variance Threshold` en conserve **20**, tandis que `SelectKBest` retient également **15 variables**. Ces différences montrent bien que les trois approches ne reposent pas sur la même logique : la sélection personnelle est guidée par l’analyse exploratoire, `Variance Threshold` par la dispersion statistique des variables, et `SelectKBest` par leur lien direct avec la variable cible.

Les fichiers `xtrain_P.csv`, `xtest_P.csv`, `xtrain_VT.csv`, `xtest_VT.csv`, `xtrain_KB.csv` et `xtest_KB.csv` constituent ainsi la sortie finale du notebook de sélection de variables. Ils serviront de base aux notebooks de modélisation, dans lesquels plusieurs combinaisons entre sélections et modèles seront comparées.

## 6. Critical discussion

Les résultats obtenus dans ce notebook montrent qu’une variable ne doit pas être jugée uniquement à partir d’un seul critère. La comparaison entre les trois méthodes de sélection met en évidence que la notion de « variable importante » dépend fortement de la méthode utilisée. `VarianceThreshold` conserve les variables les plus dispersées, mais ne tient pas compte de la survie. `SelectKBest`, au contraire, mesure le lien statistique avec la cible, mais peut sous-estimer des effets plus complexes. Enfin, la sélection personnelle dépend davantage de l’analyse exploratoire et du raisonnement de l’analyste.

Le cas de la structure familiale est particulièrement révélateur. Sous sa forme numérique brute, `family_size` apparaissait peu convaincante dans certaines sélections automatiques. Pourtant, l’analyse exploratoire montrait que la relation avec la survie n’était pas linéaire : voyager seul, en petite famille ou en grande famille ne semblait pas avoir le même effet. La création de `family_group` a permis de mieux capter cette structure. Cela montre qu’un bon feature engineering peut parfois avoir plus d’impact que le choix d’une méthode automatique de sélection.

Une autre limite importante est que plusieurs variables fortement prédictives semblent traduire un même phénomène social sous des formes différentes. `pclass`, `fare`, `has_cabin` et certains titres ne décrivent pas seulement des caractéristiques indépendantes, mais reflètent en partie le statut social des passagers. Une question importante est donc de savoir si le modèle apprend réellement des mécanismes de survie, ou s’il reconstruit surtout une hiérarchie sociale déjà présente dans les données.

## Conclusion

Dans ce notebook, nous avons comparé trois approches de sélection de variables appliquées au dataset Titanic prétraité.

La première approche, fondée sur l’analyse exploratoire de la phase 1, a permis de construire une sélection personnelle cohérente avec les constats dégagés précédemment. Elle met en avant les variables liées au sexe, à la classe, au titre, au statut social et à la structure familiale regroupée.

La deuxième approche, `Variance Threshold`, a permis d’éliminer uniquement les variables les moins dispersées, tout en conservant l’essentiel de l’information. Cette méthode reste cependant indépendante de la variable cible et ne permet donc pas, à elle seule, d’identifier les variables les plus explicatives de la survie.

La troisième approche, `SelectKBest`, a mis en évidence les variables les plus fortement liées à la survie selon un critère statistique. Elle a notamment confirmé l’importance du sexe, du titre, de la classe, du tarif, de l’information de cabine et de la structure familiale regroupée. La variable `family_group` s’est révélée particulièrement utile, car cette représentation capte mieux l’effet de la structure familiale que la seule variable numérique `family_size`.

À l’issue de ce notebook, les fichiers correspondant aux trois sélections ont été sauvegardés et pourront être utilisés dans les notebooks de modélisation. L’étape suivante consistera à combiner ces sélections avec plusieurs modèles de classification afin d’identifier la solution la plus performante.